In [ ]:
import os
import shutil
import numpy as np
from osgeo import gdal
import subprocess

def extract_and_geocode(vrt_file, output_tif):
    if not os.path.exists(vrt_file):
        print(f"  [!] Error: {vrt_file} not found. Skipping.")
        return

    print(f"\n--- Extracting Amplitude from {vrt_file} ---")
    
    # 1. Read complex SLC data and calculate magnitude (amplitude)
    ds = gdal.Open(vrt_file)
    print("  > Calculating amplitude in radar geometry...")
    comp_data = ds.GetRasterBand(1).ReadAsArray()
    amp_data = np.abs(comp_data).astype(np.float32)
    ds = None

    # 2. Disguise as topophase.cor (which ISCE knows exactly how to geocode)
    print("  > Disguising as topophase.cor for ISCE terrain-correction...")
    cor_bin = "merged/topophase.cor"
    cor_bak = "merged/topophase.cor.bak"
    
    # Backup the real coherence file
    if os.path.exists(cor_bin):
        shutil.move(cor_bin, cor_bak)
        
    # Write our amplitude into the coherence file's spot as raw binary
    with open(cor_bin, "wb") as f:
        amp_data.tofile(f)
        
    # 3. Trigger ISCE's geocode step 
    # (ISCE thinks it's geocoding coherence, but it's actually our amplitude!)
    print("  > Warping to geographic coordinates via topsApp...")
    subprocess.run(["topsApp.py", "--dostep=geocode"], stdout=subprocess.DEVNULL)
    
    # 4. Save the geocoded result as our final GeoTIFF
    print("  > Saving to GeoTIFF...")
    geo_vrt = "merged/topophase.cor.geo.vrt"
    ds_geo = gdal.Open(geo_vrt)
    driver = gdal.GetDriverByName("GTiff")
    driver.CreateCopy(output_tif, ds_geo)
    ds_geo = None
    
    # 5. Clean up and restore the real file
    os.remove(cor_bin)
    if os.path.exists(cor_bak):
        shutil.move(cor_bak, cor_bin)
        
    print(f"✅ Success! Saved {output_tif}")

if __name__ == "__main__":
    print("Starting ISCE Geocode workaround...")
    
    # In topsApp, the merged files are usually named reference.slc / secondary.slc
    ref_vrt = "merged/reference.slc.vrt"
    sec_vrt = "merged/secondary.slc.vrt"
    
    # Fallbacks in case you are using an older version of ISCE2
    if not os.path.exists(ref_vrt): ref_vrt = "merged/master.slc.vrt"
    if not os.path.exists(sec_vrt): sec_vrt = "merged/slave.slc.vrt"
    
    extract_and_geocode(ref_vrt, "reference_amplitude.tif")
    extract_and_geocode(sec_vrt, "secondary_amplitude.tif")
    
    print("\nRestoring true coherence file back to its geographic state...")
    subprocess.run(["topsApp.py", "--dostep=geocode"], stdout=subprocess.DEVNULL)
    print("All done! You now have your pure pre-event and post-event amplitudes.")

Error: File not found -> merged/secondary.slc.full.geo
